In [ ]:
import ee
from tqdm import tqdm
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import geemap
from skimage.transform import resize

# import seaborn and theme it
import seaborn as sns
sns.set_theme(style="ticks", palette="pastel", font_scale=1.5, rc={"figure.figsize": (10, 7), "lines.linewidth": 2})


In [ ]:
ee.Authenticate()
ee.Initialize(project="ee-miantsaandr")

In [ ]:
folder_path = "../data/parquet/"
path_parquet_flood = (
    folder_path + "gd_flood.parquet"
)
path_parquet_disasters = (
    folder_path + "gd_disasters.parquet"
)

gdf_flood = gpd.read_parquet(path_parquet_flood)
gdf_disasters = gpd.read_parquet(path_parquet_disasters)

In [ ]:
gdf_flood

In [ ]:
path_parquet_idu = (
    folder_path + "gd_idu_2025.parquet"
)

gdf_idu = gpd.read_parquet(path_parquet_idu)

In [ ]:
# function that plot distribution figures
def plot_distribution(gdf, title):
    gdf["Total figures"].hist(bins=50)
    plt.title(title)
    plt.xlabel("Total figures")
    plt.ylabel("Frequency")
    # log scale for better visibility
    plt.yscale("log")
    plt.grid()
    plt.show()

plot_distribution(gdf_flood, "Figure Distribution")
plot_distribution(gdf_disasters, "Figure Distribution")

In [ ]:
def resize_image(img_arr, target_size):
    return resize(img_arr, target_size, anti_aliasing=False)

In [ ]:
def scale_landsat(img):
    optical = img.select("SR_B.").multiply(0.0000275).add(-0.2)
    return img.addBands(optical, None, True)


def cloud_mask_landsat9(image):
    qa = image.select("QA_PIXEL")
    cloud_mask = qa.bitwiseAnd(1 << 3).eq(0).And(qa.bitwiseAnd(1 << 4).eq(0))
    return image.updateMask(cloud_mask)

def mask_viirs_clouds(image):
    cloud_mask = image.select("QF_Cloud_Mask")
    cloud_confidence = cloud_mask.rightShift(6).bitwiseAnd(3)
    clear_mask = cloud_confidence.lte(1)
    return image.updateMask(clear_mask)

def ensure_crs_alignment(gdf, target_crs="EPSG:4326"):
    """Ensure GeoDataFrame is in the target CRS (WGS84 by default for Earth Engine)"""
    if gdf.crs != target_crs:
        print(f"Converting CRS from {gdf.crs} to {target_crs}")
        gdf = gdf.to_crs(target_crs)
    return gdf

def create_matched_geometry(point, buffer_distance, target_crs="EPSG:4326"):
    """Create Earth Engine geometry with proper CRS handling"""
    # Ensure point is in WGS84 for Earth Engine
    if hasattr(point, "to_crs"):
        point_wgs84 = point.to_crs(target_crs)
        ee_point = ee.Geometry.Point([point_wgs84.geometry.x, point_wgs84.geometry.y])
    else:
        ee_point = ee.Geometry.Point([point.geometry.x, point.geometry.y])

    buffer_geom = ee_point.buffer(buffer_distance)
    bounds_rect = buffer_geom.bounds()

    return ee_point, buffer_geom, bounds_rect


In [ ]:
def get_landsat9_rgb_median_numpy(
    point_index,
    gdf,
    buffer_distance=5000,
    start_or_end="Start date",
    scale=30,
    target_size=(330, 330),
    target_crs="EPSG:4326",
):
    gdf_aligned = ensure_crs_alignment(gdf, target_crs)

    point = gdf_aligned.iloc[point_index]
    ee_point, buffer_geom, bounds_rect = create_matched_geometry(point, buffer_distance)

    # start and end dates (30 around the event date)
    start_date = point['Start date']
    end_date = point['End date']
    if isinstance(start_date, str) or isinstance(end_date, str):
        start_date = pd.to_datetime(start_date)
        end_date = pd.to_datetime(end_date)
    date = start_date + (end_date - start_date) / 2  # Use midpoint for the date
    date_start = (date - pd.Timedelta(days=90)).strftime("%Y-%m-%d")
    date_end = (date + pd.Timedelta(days=90)).strftime("%Y-%m-%d")

    collection = (
        ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
        .filterDate(date_start, date_end)
        .filterBounds(bounds_rect)
        .filter(ee.Filter.lt("CLOUD_COVER", 60))
        .map(cloud_mask_landsat9)
        .map(scale_landsat)
    )

    collection_size = collection.size().getInfo()
    if collection_size == 0:
        print(
            f"No Landsat images found for point {point_index} between {date_start} and {date_end}"
        )
        return None, None

    composite = collection.median().clip(bounds_rect)
    rgb = composite.select(["SR_B4", "SR_B3", "SR_B2"])
    date_str = f"{date_start}_to_{date_end}"

    try:
        np_image = geemap.ee_to_numpy(
            rgb, region=bounds_rect, scale=scale
        )

        if np_image.ndim != 3:
            print(
                f"Unexpected image dimensions for point {point_index}: {np_image.shape}"
            )
            return None, None

        # Resize image
        resized = resize_image(np_image, target_size=target_size)

        return resized, date_str

    except Exception as e:
        print(f"Error processing Landsat data for point {point_index}: {str(e)}")
        return None, None

In [ ]:
# test get landsat9_rgb_median_numpy
test_index = np.random.randint(0, len(gdf_flood))
resized_img, date_label = get_landsat9_rgb_median_numpy(
    test_index, gdf_flood, scale=30
)
# Plotting the resized image
print(f"Resized image shape: {resized_img.shape}")
print(f"Date label: {date_label}")

# Create figure
plt.figure(figsize=(10, 8))

# Enhance the RGB visualization with contrast stretching
# Landsat typically needs enhancement for better visualization
rgb_enhanced = np.copy(resized_img)
# Apply contrast stretching - multiply by factor to enhance visibility
# but keep within valid range [0,1]
rgb_enhanced = np.clip(rgb_enhanced * 3.5, 0, 1)  

# Display the image with enhanced colors
plt.imshow(rgb_enhanced)
plt.title(f"Landsat 9 RGB on {date_label}", fontsize=14)
plt.axis("off")

# Add colorbar to show scale
plt.colorbar(label="Scaled Reflectance", shrink=0.7)
plt.tight_layout()
plt.show()

In [ ]:
def get_viirs_nightlight_median_numpy(
    point_index,
    gdf,
    buffer_distance=5000,
    start_or_end="Start date",
    scale=500,
    target_size=(20, 20),
    target_crs="EPSG:4326",
):

    # Ensure CRS alignment
    gdf_aligned = ensure_crs_alignment(gdf, target_crs)

    # Extract point and dates
    point = gdf_aligned.iloc[point_index]
    ee_point, buffer_geom, bounds_rect = create_matched_geometry(point, buffer_distance)

    date = point[start_or_end]
    if isinstance(date, str):
        date = pd.to_datetime(date)

    # two weeks around the event date
    date_start = (date - pd.Timedelta(days=7)).strftime("%Y-%m-%d")
    date_end = (date + pd.Timedelta(days=7)).strftime("%Y-%m-%d")

    # Image collection: filtered, cloud-masked
    collection = (
        ee.ImageCollection("NASA/VIIRS/002/VNP46A2")
        .filterDate(date_start, date_end)
        .filterBounds(bounds_rect)
        .select("Gap_Filled_DNB_BRDF_Corrected_NTL", "QF_Cloud_Mask", "Snow_Flag")
        .map(mask_viirs_clouds)
    )

    # Check if collection has images
    collection_size = collection.size().getInfo()
    if collection_size == 0:
        print(
            f"No VIIRS images found for point {point_index} between {date_start} and {date_end}"
        )
        return None, None

    # Median composite
    composite = collection.median().clip(bounds_rect)
    date_str = f"{date_start}_to_{date_end}"

    try:
        # Convert to NumPy array
        np_image = geemap.ee_to_numpy(
            composite.select("Gap_Filled_DNB_BRDF_Corrected_NTL"),
            region=bounds_rect,
            scale=scale,
        )

        # Handle potential 3D array
        if np_image.ndim == 3 and np_image.shape[-1] == 1:
            np_image = np_image[:, :, 0]

        # Resize image
        resized = resize_image(np_image, target_size=target_size)

        return resized, date_str

    except Exception as e:
        print(f"Error processing VIIRS data for point {point_index}: {str(e)}")
        return None, None

In [ ]:
# plot viirs nightlight
# test_index = np.random.randint(0, len(gdf_flood))
test_index = 40
resized_img_viirs, date_label_viirs = get_viirs_nightlight_median_numpy(
    test_index, gdf_flood, buffer_distance=5000, scale=500, target_size=(20, 20)
)
# Plotting the resized image
print(f"Resized VIIRS image shape: {resized_img_viirs.shape}")
print(f"Date label: {date_label_viirs}")
# Create figure
plt.figure(figsize=(10, 8))
# Enhance the nightlight visualization with contrast stretching
# VIIRS typically needs enhancement for better visualization
viirs_enhanced = np.copy(resized_img_viirs)
# Display the image with enhanced colors
plt.imshow(viirs_enhanced)
plt.title(f"VIIRS Nightlight on {date_label_viirs}", fontsize=14)
plt.axis("off")
# Add colorbar to show scale
plt.colorbar(label="Scaled Radiance", shrink=0.7)
plt.tight_layout()
plt.show()

In [ ]:
def collect_unified_data(
    point_indices,
    gdf,
    buffer_distance=5000,
    start_or_end="Start date",
    viirs_scale=500,
    landsat_scale=30,
    # target_size=(330, 330),
    target_crs="EPSG:4326",
):

    results = {}

    for idx in point_indices:

        # Get nighttime lights data for start and end dates
        viirs_data_start = get_viirs_nightlight_median_numpy(
            idx,
            gdf,
            buffer_distance,
            "Start date",
            viirs_scale,
            (20, 20),
            target_crs,
        )

        viirs_data_end = get_viirs_nightlight_median_numpy(
            idx,
            gdf,
            buffer_distance,
            "End date",
            viirs_scale,
            (20, 20),
            target_crs,
        )

        # Get RGB data
        rgb_data = get_landsat9_rgb_median_numpy(
            idx,
            gdf,
            buffer_distance,
            start_or_end,
            landsat_scale,
            (330, 330),
            target_crs,
        )

        results[idx] = {
            "viirs_start": viirs_data_start[0],
            "viirs_end": viirs_data_end[0],
            "rgb": rgb_data[0],
            "figures": gdf.iloc[idx]["Total figures"],
            'iso3': gdf.iloc[idx]["ISO3"],
        }

    return results

- Data collect
    - parquet doesn't work because it doesn't save multi arrays
    

In [ ]:
# collect some point
num = 5
point_indices = range(num)
collected_data = collect_unified_data(
    point_indices,
    gdf_flood,
    buffer_distance=5000,
    start_or_end="Start date",
    viirs_scale=500,
    landsat_scale=30,
    # target_size=(330, 330),
    target_crs="EPSG:4326",
)

# view in dataframe
df_collected = pd.DataFrame.from_dict(
    collected_data, orient="index", columns=["viirs_start", "viirs_end", "rgb", "figures", "iso3"]
)
# Display the first few rows of the DataFrame
df_collected

In [ ]:
# shape of the things in the dataframe
print(f"Shape of VIIRS Start: {df_collected['viirs_start'].iloc[0].shape}")
print(f"Shape of VIIRS End: {df_collected['viirs_end'].iloc[0].shape}")
print(f"Shape of RGB: {df_collected['rgb'].iloc[0].shape}")
print(f"Figures: {df_collected['figures'].iloc[0]}")
print(f"ISO3: {df_collected['iso3'].iloc[0]}")


In [ ]:
def plot_collected_data(data_row, index, rgb_enhance_factor=3.5):
    fig, axs = plt.subplots(1, 3, figsize=(15, 5))

    # Plot VIIRS Nightlight Start
    axs[0].imshow(data_row["viirs_start"], cmap="gray")
    axs[0].set_title(f"VIIRS Nightlight Start (Index {index})")
    axs[0].axis("off")

    # Plot VIIRS Nightlight End
    axs[1].imshow(data_row["viirs_end"], cmap="gray")
    axs[1].set_title(f"VIIRS Nightlight End (Index {index})")
    axs[1].axis("off")

    # Enhance RGB image
    rgb = np.copy(data_row["rgb"])
    rgb_enhanced = np.clip(rgb * rgb_enhance_factor, 0, 1)

    # Plot RGB Image
    axs[2].imshow(rgb_enhanced)
    axs[2].set_title(f"RGB Image (Index {index})")
    axs[2].axis("off")

    # Main Title with ISO3
    fig.suptitle(
        f"ISO3: {data_row['iso3']} — Total Figures: {data_row['figures']}", fontsize=16
    )

    plt.tight_layout()
    plt.show()


In [ ]:
# Plot the random example from the collected data
random_index = np.random.choice(df_collected.index)
plot_collected_data(df_collected.loc[random_index], random_index)

In [ ]:
import h5py
import numpy as np


def batched(iterable, n):
    """Yield successive n-sized chunks from iterable."""
    for i in range(0, len(iterable), n):
        yield iterable[i : i + n]

In [ ]:
# read and put in dataframe
def read_hdf5_to_dataframe(h5_path="unified_data.h5"):
    with h5py.File(h5_path, "r") as f:
        viirs_start = f["viirs_start"][:]
        viirs_end = f["viirs_end"][:]
        rgb = f["rgb"][:]
        figures = f["figures"][:]
        indices = f["indices"][:]
        iso3 = f["iso3"][:]

    # Decode bytes to strings
    iso3_decoded = [x.decode("utf-8") if isinstance(x, bytes) else x for x in iso3]

    # Create a DataFrame
    df = pd.DataFrame({
        "index": indices,
        "viirs_start": list(viirs_start),
        "viirs_end": list(viirs_end),
        "rgb": list(rgb),
        "figures": figures,
        "iso3": iso3_decoded
    })

    return df

In [ ]:
# # plot example
def plot_hdf5_example(data_row, index):
    fig, axs = plt.subplots(1, 3, figsize=(15, 5))

    # Plot VIIRS Nightlight Start
    axs[0].imshow(data_row["viirs_start"], cmap="gray")
    axs[0].set_title(f"VIIRS Nightlight Start (Index {index})")
    axs[0].axis("off")

    # Plot VIIRS Nightlight End
    axs[1].imshow(data_row["viirs_end"], cmap="gray")
    axs[1].set_title(f"VIIRS Nightlight End (Index {index})")
    axs[1].axis("off")

    # Plot RGB Image
    axs[2].imshow(data_row["rgb"])
    axs[2].set_title(f"RGB Image (Index {index})")
    axs[2].axis("off")
    plt.suptitle(f"Total Figures: {data_row['figures']}", fontsize=16)

    plt.tight_layout()
    plt.show()

# # Plot a random example from the DataFrame
# random_index = np.random.choice(df_unified.index)
# plot_hdf5_example(df_unified.loc[random_index], random_index)

- concurent future

In [ ]:
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed


def safe_collect(batch, gdf):
    try:
        return batch, collect_unified_data(batch, gdf)
    except Exception as e:
        print(f"Error in batch {batch}: {e}")
        return batch, {}


def write_hdf5_batches_parallel(
    point_indices, gdf, h5_path="output_parallel.h5", batch_size=32, max_workers=8
):
    with h5py.File(h5_path, "w") as f:
        # Create resizable datasets with float RGB
        viirs_start_ds = f.create_dataset(
            "viirs_start",
            shape=(0, 20, 20),
            maxshape=(None, 20, 20),
            dtype="float32",
            chunks=True,
        )
        viirs_end_ds = f.create_dataset(
            "viirs_end",
            shape=(0, 20, 20),
            maxshape=(None, 20, 20),
            dtype="float32",
            chunks=True,
        )
        rgb_ds = f.create_dataset(
            "rgb",
            shape=(0, 330, 330, 3),
            maxshape=(None, 330, 330, 3),
            dtype="float32",
            chunks=True,
        )
        figures_ds = f.create_dataset(
            "figures", shape=(0,), maxshape=(None,), dtype="float32", chunks=True
        )
        indices_ds = f.create_dataset(
            "indices", shape=(0,), maxshape=(None,), dtype="int32", chunks=True
        )

        iso3_dtype = h5py.string_dtype(encoding='utf-8', length=3)
        iso3_ds = f.create_dataset(
            "iso3", shape=(0,), maxshape=(None,), dtype=iso3_dtype, chunks=True
        )

        current_idx = 0
        futures = []
        batches = list(batched(point_indices, batch_size))

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            for batch in batches:
                futures.append(executor.submit(safe_collect, batch, gdf))

            for future in tqdm(
                as_completed(futures), total=len(futures), desc="Processing"
            ):
                batch, data = future.result()

                valid_entries = [
                    (idx, entry)
                    for idx, entry in data.items()
                    if all(
                        entry.get(key) is not None
                        for key in ["viirs_start", "viirs_end", "rgb"]
                    )
                ]

                if not valid_entries:
                    continue

                indices, valid_data = zip(*valid_entries)

                viirs_start = np.stack([entry["viirs_start"] for entry in valid_data])
                viirs_end = np.stack([entry["viirs_end"] for entry in valid_data])
                rgb = np.stack(
                    [entry["rgb"] for entry in valid_data]
                )  # float32 expected
                figures = np.array(
                    [entry["figures"] for entry in valid_data], dtype="float32"
                )
                idx_array = np.array(indices, dtype="int32")
                iso3_array = np.array([entry["iso3"] for entry in valid_data], dtype=iso3_dtype)

                new_size = current_idx + len(valid_data)

                # Resize HDF5 datasets
                viirs_start_ds.resize(new_size, axis=0)
                viirs_end_ds.resize(new_size, axis=0)
                rgb_ds.resize(new_size, axis=0)
                figures_ds.resize(new_size, axis=0)
                indices_ds.resize(new_size, axis=0)
                iso3_ds.resize(new_size, axis=0)

                # Write batch
                viirs_start_ds[current_idx:new_size] = viirs_start
                viirs_end_ds[current_idx:new_size] = viirs_end
                rgb_ds[current_idx:new_size] = rgb
                figures_ds[current_idx:new_size] = figures
                indices_ds[current_idx:new_size] = idx_array
                iso3_ds[current_idx:new_size] = iso3_array

                current_idx = new_size


In [ ]:
gdf_disasters

- Code used to collect dissaster images

In [ ]:
# point_indices = list(range(len(gdf_disasters)))  # Use indices of gdf_disasters
# # ten only
# # point_indices = list(range(10)) 

# write_hdf5_batches_parallel(
#     point_indices=point_indices,
#     gdf=gdf_disasters, # Use gdf_disasters for disaster data
#     h5_path="disaster.h5", # output file name
#     batch_size=32,
# )

In [ ]:
# df_unified_parallel = read_hdf5_to_dataframe(h5_path="disaster.h5")

In [ ]:
# df_unified_parallel.head()

In [ ]:
# len(df_unified_parallel)

In [ ]:
# type(df_unified_parallel.iloc[0]["iso3"])  # Check type of iso3 field

In [ ]:
# # plot a random example from the parallel dataframe
# random_index = np.random.choice(df_unified_parallel.index)
# plot_hdf5_example(df_unified_parallel.loc[random_index], random_index)


In [ ]:
# modify so indices is the index of the dataframe
def read_hdf5_to_dataframe_with_index(h5_path="unified_parallel.h5"):
    with h5py.File(h5_path, "r") as f:
        viirs_start = f["viirs_start"][:]
        viirs_end = f["viirs_end"][:]
        rgb = f["rgb"][:]
        figures = f["figures"][:]
        indices = f["indices"][:]
        iso3 = f["iso3"][:]
    
    # Decode bytes to strings for iso3
    iso3_decoded = [x.decode("utf-8") if isinstance(x, bytes) else x for x in iso3]

    # Create a DataFrame with indices as the index
    df = pd.DataFrame({
        "viirs_start": list(viirs_start),
        "viirs_end": list(viirs_end),
        "rgb": list(rgb),
        "figures": figures,
        "iso3": iso3_decoded
    }, index=indices)

    df.sort_index(inplace=True)  # Ensure indices are sorted

    return df

# Read the HDF5 file into a DataFrame with indices
df_unified_with_index = read_hdf5_to_dataframe_with_index(h5_path="disaster.h5")

In [ ]:
# df_unified_with_index.sort_index(inplace=True)

In [ ]:
# df_unified_with_index

In [ ]:
# load gd_idu_2025
path_parquet_idu = (
    folder_path + "gd_idu_2025.parquet"
)
gdf_idu = gpd.read_parquet(path_parquet_idu)

In [ ]:
gdf_idu

In [ ]:
point_indices_u = list(range(len(gdf_idu)))  # Use indices of gdf_idu

write_hdf5_batches_parallel(
    point_indices=point_indices_u,
    gdf=gdf_idu,
    h5_path="testing.h5",
    batch_size=32,
)


In [ ]:
idu_unified_with_index = read_hdf5_to_dataframe_with_index(h5_path="testing.h5")


In [ ]:
idu_unified_with_index.sort_index(inplace=True)

In [ ]:
idu_unified_with_index.head()